# Dynamic Simulation of Two Rooms
## MSE course EVA_CE4IB 2026

Authors: Drigatti Alessio, Zurbrügg Milena

This notebook models two adjacent rooms at the corner of a building as a lumped-parameter thermal network of thermal resistances and capacities. The network is converted into a state-space model and checked with a steady-state analysis. It is then simulated over one year with weather data from Hombrechtikon (ZH). The main results are the indoor air temperatures of both rooms and the heating and cooling power needed to keep Room 5 at its setpoint.

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/drigaa/rEVA_CE4IB/HEAD?urlpath=%2Fdoc%2Ftree%2Fcasestudy.ipynb)

## Project Overview
The objective of this project is to simulate the thermal dynamics of two rooms located at the corner of a building. Figure 1 shows the floor plan of the area under consideration, including the relevant dimensions. Each room has a height of 3 m.

![thermal-network](images/Room_Plan.png)

Figure 1: Floor plan of the considered area.

The two grey areas represent the rooms considered in this project. Room 5, on the right-hand side, is located at the corner of the building and therefore has two external façades. One of them is an insulated concrete wall (orange) facing east, the other contains a window (blue) facing north. Room 5 is separated from Room 9 by a non-insulated concrete wall (orange).

Room 9 is connected to the exterior only through a single north-facing window (blue). The walls highlighted in red are assumed to have no influence on the thermal behaviour of the rooms: the spaces behind them are assumed to be at the same temperature as the rooms, so no heat flows through them and they are neglected in the analysis. The same assumption is made for the floor and the ceiling of the rooms.

### Main assumptions
- For simplicity, every wall and window is modelled as a square surface of $l \times l = 3\,\text{m} \times 3\,\text{m} = 9\,\text{m}^2$. The windows therefore cover the whole façade.
- The floor, the ceiling and the walls marked in red are adiabatic (no heat flow).
- The air in each room is perfectly mixed and has a single temperature.
- Long-wave radiation between the interior surfaces is neglected, i.e. all surfaces of a room are assumed to have approximately the same temperature (see Outlook).
- Ventilation and infiltration are not considered; internal heat gains are constant.
- Material properties and heat transfer coefficients are constant.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

import dm4bem as db

In [ ]:
from settings_simulation import *

## Thermal Network

The thermal behaviour of the two rooms is represented by a lumped-parameter thermal network consisting of thermal conductances, thermal capacities, temperature sources and heat flow sources. Figure 2 shows the complete network of the considered area.

![thermal-network](images/Screenshot%202026-10-29%20093356.png)

Figure 2: Thermal network for the area shown in Figure 1.

The outdoor temperature $T_o$ acts as the external boundary condition for all building envelope components (it should not be confused with $\theta_0$, the temperature of node 0). The room air temperatures are represented by nodes 5 and 9, corresponding to Rooms 5 and 9 in the floor plan.

Each room air node is connected through a conductance (branch 11 for Room 5, branch 10 for Room 9) to a temperature source $T_{i,sp}$, the indoor setpoint temperature. Together, they model the HVAC system as a proportional (P) controller: the heating or cooling power delivered to the room is

\begin{equation}
q_{HVAC} = K_p \, (T_{i,sp} - \theta),
\end{equation}

where $K_p$ (W/K) is the controller gain. A gain of $K_p \to \infty$ corresponds to perfect control ($\theta = T_{i,sp}$), while $K_p = 0$ corresponds to a free-floating building without HVAC.

### Exterior Corner Wall of Room 5

The insulated exterior wall of Room 5 is shown in orange in Figure 2 and covers nodes 0 to 4. Heat is transferred from the outdoor air to the outer wall surface (node 0) by convection (branch 0). Solar radiation absorbed by the outer wall surface is represented by the heat flow source $\Phi_o$ acting on node 0.

The wall consists of an outer insulation layer and an inner concrete layer. Each layer is divided into two halves, which gives four conductive branches: branches 1 and 2 for the insulation and branches 3 and 4 for the concrete. The thermal capacity of each layer is placed in its middle node (node 1 for the insulation, node 3 for the concrete). Node 4 is the inner wall surface, which is connected to the air of Room 5 (node 5) by convection (branch 5).

### Internal Wall Between the Rooms

The wall separating Rooms 5 and 9 consists of non-insulated concrete and is represented by nodes 6 to 8, where nodes 6 and 8 are its two surfaces. Convective heat transfer occurs on both surfaces (branches 6 and 9), while conduction through the concrete is described by branches 7 and 8. A thermal capacity is assigned to node 7 to account for heat storage in the wall.

This wall thermally couples the two rooms: heat flows through it whenever the temperatures of nodes 5 and 9 differ.

### Room Air Nodes

The thermal capacity of the indoor air of each room can be assigned to nodes 5 and 9. It is switched on or off with the parameter `air_with_capacity` in the settings file.

Both rooms are subjected to internal heat gains $\dot Q_{aux}$ (`Qa` in the code), representing occupants, lighting, electrical devices and other internal heat sources. These gains are added directly to the room air nodes.

Solar radiation transmitted through the windows is absorbed by the interior wall surfaces: nodes 4 and 6 in Room 5 and node 8 in Room 9. It is represented by the heat flow source $\Phi_i$ acting on these nodes.

### Windows

Each room has a window exposed to the outdoor environment. Heat transfer through the windows is represented by branches 13 (Room 5) and 12 (Room 9). They connect the outdoor temperature $T_o$ directly to the room air nodes through an equivalent conductance based on the U-value of the window. The thermal capacity of the windows is neglected.

**!!!** **IMPORTANT** **!!!**

Change the simulation settings in [settings_simulation.py](settings_simulation.py) and always re-run the whole notebook afterwards.

| Parameter | Meaning |
|---|---|
| `controller`, `Kp_value` | Activate the P-controller (HVAC) and set its gain $K_p$ in W/K |
| `imposed_time_step`, `Δt` | Use a fixed time step `Δt` (s) instead of the one derived from the eigenvalues |
| `temp_init` | Initial temperature of all nodes, °C |
| `temp_indor_setpoint` | Indoor setpoint temperature $T_{i,sp}$, °C |
| `Qa` | Internal heat gains per room $\dot Q_{aux}$, W |
| `start_date`, `end_date` | Simulation period |
| `air_with_capacity` | Include the thermal capacity of the room air |
| `To_ss`, `Ti_sp_ss`, `Qa_ss` | Constant inputs used for the steady-state analysis |
| `slope`, `azimuth_east`, `azimuth_north`, `latitude` | Orientation of the surfaces and latitude of the site |
| `wall_area`, `albedo`, `alpha`, `beta_window`, `view_factor` | Parameters for the solar heat gains |

In [ ]:
number_of_nodes = 10
number_of_branches = 14

# thermal properties of air
air = {
    'Density': 1.2,             # kg/m³
    'Specific heat': 1000       # J/(kg·K)
}

# convection coefficients, W/(m²·K)
h = pd.DataFrame([{'in': 4., 'out': 10.}], index=['h'])

# side length of the rooms, m (all walls and windows are l x l)
l = 3

# inner layer of the exterior wall and internal wall
concrete = {
    'Conductivity': 1.400,      # W/(m·K)
    'Density': 2300.0,          # kg/m³
    'Specific heat': 880,       # J/(kg·K)
    'Width': 0.15,              # m
    'Surface': l ** 2           # m²
}

# outer layer of the exterior wall
insulation = {
    'Conductivity': 0.040,
    'Density': 16.0,
    'Specific heat': 1210,
    'Width': 0.25,
    'Surface': l ** 2           # m²
}

# windows, U-value in W/(m²·K)
glass = {
    'U-Value': 0.6,
    'Surface': l ** 2           # m²
}

wall = pd.DataFrame.from_dict(
    {
        'Layer_out': insulation,
        'Layer_in': concrete,
        'window': glass
    },
    orient='index')

wall

## System of Differential-Algebraic Equations (DAE)
### Temperature nodes and flow branches

The thermal network is described by two sets of equations. The heat flow in each branch is given by the temperature difference across the branch:

\begin{equation}
q = G\,(-A\,\theta + b),
\end{equation}

and the energy balance of each node is

\begin{equation}
C\,\dot\theta = A^T q + f,
\end{equation}

where $A$ is the incidence matrix, $G$ the diagonal conductance matrix, $C$ the diagonal capacity matrix, $b$ the vector of temperature sources in the branches and $f$ the vector of heat flow sources in the nodes. Because some nodes have no thermal capacity ($C_{jj} = 0$), part of the second set of equations is algebraic instead of differential, which is why the system is a DAE.

$q$ is the vector of heat flows through the 14 branches,

\begin{equation}
q =
\begin{bmatrix}
q_0 & q_1 & q_2 & q_3 & q_4 & q_5 &
q_6 & q_7 & q_8 & q_9 & q_{10} & q_{11} & q_{12} & q_{13}
\end{bmatrix}
^{T},
\end{equation}

and the temperature vector

\begin{equation}
\theta =
\begin{bmatrix}
\theta_0 & \theta_1 & \theta_2 & \theta_3 &
\theta_4 & \theta_5 & \theta_6 & \theta_7 & \theta_8 & \theta_9
\end{bmatrix}^{T}
\end{equation}

contains the temperatures of the ten nodes.

In [ ]:
θ = [f'θ{i}' for i in range(number_of_nodes)]
q = [f'q{i}' for i in range(number_of_branches)]

#### Incidence matrix $A$
The incidence matrix describes how the 14 branches (rows) are connected to the 10 nodes (columns). An entry of $1$ means that the flow of the branch enters the node, and $-1$ means that it leaves the node. Branches 0 and 10–13 are connected to only one node, because they start at a temperature source. The chosen flow directions are only a sign convention: a negative computed flow means that the heat flows in the opposite direction.

In [ ]:
A = np.zeros((number_of_branches, number_of_nodes))

# exterior wall: outdoor -> θ0 -> θ1 -> θ2 -> θ3 -> θ4 -> room 5
A[0, 0] = 1
A[1, 0], A[1, 1] = -1, 1
A[2, 1], A[2, 2] = -1, 1
A[3, 2], A[3, 3] = -1, 1
A[4, 3], A[4, 4] = -1, 1
A[5, 4], A[5, 5] = -1, 1
# internal wall: room 9 -> θ8 -> θ7 -> θ6 -> room 5
A[6, 6], A[6, 5] = -1, 1
A[7, 7], A[7, 6] = -1, 1
A[8, 8], A[8, 7] = -1, 1
A[9, 9], A[9, 8] = -1, 1
# HVAC controllers: room 9 (q10), room 5 (q11)
A[10, 9] = 1
A[11, 5] = 1
# windows: room 9 (q12), room 5 (q13)
A[12, 9] = 1
A[13, 5] = 1

A = pd.DataFrame(A, index=q, columns=θ)

### Conductance Matrix $G$

The diagonal conductance matrix contains the thermal conductances (W/K) of all branches:

- **Conduction** (branches 1–4, 7, 8): each layer is split into two halves, so $G = \dfrac{\lambda}{w/2}\,S$, with the thermal conductivity $\lambda$, the layer width $w$ and the surface area $S$.
- **Convection** (branches 0, 5, 6, 9): $G = h\,S$, with $h_{out} = 10$ W/(m²·K) outside and $h_{in} = 4$ W/(m²·K) inside.
- **Windows** (branches 12, 13): $G = U\,S$. The U-value already includes the inner and outer surface resistances, so no separate convection branches are needed.
- **HVAC controller** (branches 10, 11): $G = K_p$. Here it is set to 0 and later overwritten with `Kp_value` if `controller = True`.

The [U-value](https://www.neuffer.de/en/u-values.php) describes the rate at which heat flows through a building component per unit area and per kelvin of temperature difference. A smaller U-value means better thermal insulation and lower heat loss. Modern triple-glazed windows typically reach U-values of about 0.4 to 0.8 W/(m²·K); the value of 0.6 W/(m²·K) used here corresponds to very good thermal performance by current standards.

The resulting values show that the insulation (branches 1 and 2) dominates the thermal resistance of the exterior wall. The whole wall has an equivalent conductance of only about 1.3 W/K, while each window has 5.4 W/K. The windows are therefore the main path for heat losses.

In [ ]:
## P-controller gain in W/K
## Will be overwritten later in the code (see settings: controller, Kp_value)
Kp = 0

G = np.zeros(number_of_branches)

# conduction: G = λ / (w/2) * S
G[[1, 2]] = (wall["Conductivity"] / (wall["Width"] / 2) * wall["Surface"]).Layer_out
G[[3, 4, 7, 8]] = (wall["Conductivity"] / (wall["Width"] / 2) * wall["Surface"]).Layer_in

# convection: G = h * S
G[[0]] = h['out'] * wall["Surface"].Layer_out
G[[5, 6, 9]] = h['in'] * wall["Surface"].Layer_in

# windows: G = U * S
G[[12, 13]] = (wall['U-Value'] * wall["Surface"]).window

# HVAC controllers
G[[10, 11]] = Kp
G = pd.Series(G, index=q)
G

### Capacity Matrix $C$

The dynamic behaviour of the system is determined by the diagonal capacity matrix

\begin{equation}
\mathbf C =
\operatorname{diag}
(0, C_1, 0, C_3, 0, C_5, 0, C_7, 0, C_9),
\end{equation}

which contains the thermal capacities (J/K) of the nodes. The capacity of a layer, $C = \rho\,c\,w\,S$, is placed in its middle node: node 1 for the insulation and node 3 for the concrete of the exterior wall, node 7 for the internal wall. The air capacities $C_5 = C_9 = \rho_{air}\,c_{air}\,l^3$ are only included if `air_with_capacity = True`; otherwise they are zero.

Nodes with zero capacity are assumed to store no heat, so their energy balance is algebraic. The insulation stores very little heat ($C_1 \approx 0.04$ MJ/K) compared to the concrete layers ($C_3 = C_7 \approx 2.7$ MJ/K).

In [ ]:
C = np.zeros(number_of_nodes)

# C = w * S * ρ * c (placed in the middle node of each layer)
C[1] = (wall['Width'] * wall['Surface'] * wall['Density'] * wall["Specific heat"]).Layer_out
C[[3, 7]] = (wall['Width'] * wall['Surface'] * wall['Density'] * wall["Specific heat"]).Layer_in

if air_with_capacity:
    C[[5, 9]] = air["Density"] * l ** 3 * air["Specific heat"]

C = pd.Series(C, index=θ)
C

### Temperature source vector $b$

The vector of temperature sources

\begin{equation}
b =
\begin{bmatrix}
T_o & 0 & 0 & 0 & 0 & 0 & 0 & 0 &
0 & 0 & T_{i,sp} & T_{i,sp} & T_o & T_o
\end{bmatrix}^{T}
\end{equation}

contains the prescribed temperatures in the branches: the outdoor temperature $T_o$ (branches 0, 12 and 13) and the indoor setpoint temperature $T_{i,sp}$ (branches 10 and 11). In the code, the non-zero entries are names (strings). They are later replaced by the corresponding time series of the input data.

In [ ]:
b = pd.Series(np.zeros(number_of_branches), index=q)
b[[0, 12, 13]] = "T_0"
b[[10, 11]] = "Ti_sp"
b

### Heat flow source vector $f$

Finally, the vector of heat flow sources

\begin{equation}
f =
\begin{bmatrix}
\Phi_o &
0 &
0 &
0 &
\Phi_i &
\dot Q_{aux} &
\Phi_i &
0 &
\Phi_i &
\dot Q_{aux}
\end{bmatrix}^{T}
\end{equation}

contains the heat flows entering the nodes directly:

- $\Phi_o$: solar radiation absorbed by the outer surface of the east wall (node 0),
- $\Phi_i$: solar radiation transmitted through the windows and absorbed by the interior wall surfaces (nodes 4 and 6 in Room 5, node 8 in Room 9),
- $\dot Q_{aux}$: internal heat gains in the room air (nodes 5 and 9), named `Qa` in the code.

The heating and cooling power of the HVAC system is not part of $f$. It follows from the controller branches 10 and 11.

In [ ]:
f = pd.Series(np.zeros(number_of_nodes), index=θ)
f[[0]] = 'Φo'
f[[5, 9]] = 'Qa'
f[[4, 6, 8]] = 'Φi'
f

### Output vector $y$
The output vector $y$ selects the node temperatures of interest (1 = output, 0 = not an output). In this work, $y$ consists of the air temperatures of Room 5 and Room 9 (nodes 5 and 9), which are the main variables for evaluating the thermal behaviour of the two rooms.

All matrices and vectors are then collected in the dictionary `TC`, which describes the complete thermal network.

In [ ]:
y = pd.Series(np.zeros(number_of_nodes), index=θ)
y[[5, 9]] = 1
y

In [ ]:
# thermal circuit
TC = {"A": A,
      "G": G,
      "C": C,
      "b": b,
      "f": f,
      "y": y}

## Steady-State Analysis from the DAE

In steady state, the temperatures do not change ($\dot\theta = 0$). Combining the two DAE equations then gives

\begin{equation}
\theta = (A^T G A)^{-1} (A^T G\, b + f).
\end{equation}

The steady-state inputs are constant values from the settings (`To_ss`, `Ti_sp_ss`, `Qa_ss`); solar radiation is set to zero.

First, the controller gain is set (if `controller = True`). The thermal network is then converted into its state-space representation with `db.tc2ss`, which is used for the second steady-state check and for the dynamic simulation.

In [ ]:
# activate the P-controller (HVAC) in branches 10 and 11
if controller:
    TC["G"][["q10", "q11"]] = Kp_value

In [ ]:
# state-space representation of the thermal network
[As, Bs, Cs, Ds, us] = db.tc2ss(TC)

##### Steady state without heat flow sources ($f = 0$) and with internal heat gains ($f = \dot Q_{aux}$ on the air nodes)
Whether the HVAC system is active depends on the setting `controller`.

In [ ]:
b_ss = np.zeros(number_of_branches)
b_ss[[0, 12, 13]] = To_ss
b_ss[[10, 11]] = Ti_sp_ss

f_ss = np.zeros(number_of_nodes)

A = TC['A']
G = TC['G']
diag_G = pd.DataFrame(np.diag(G), index=G.index, columns=G.index)

θss = np.linalg.inv(A.T @ diag_G @ A) @ (A.T @ diag_G @ b_ss + f_ss)
print("Results without heat flow sources (f = 0)")
print(f'θss = {np.around(θss, 2)} °C')

In [ ]:
b_ss = np.zeros(number_of_branches)
b_ss[[0, 12, 13]] = To_ss
b_ss[[10, 11]] = Ti_sp_ss

f_ss = np.zeros(number_of_nodes)
f_ss[[5, 9]] = Qa_ss

A = TC['A']
G = TC['G']
diag_G = pd.DataFrame(np.diag(G), index=G.index, columns=G.index)

θss_Q = np.linalg.inv(A.T @ diag_G @ A) @ (A.T @ diag_G @ b_ss + f_ss)
print(f"Results with auxiliary heat source {Qa_ss} Watts on node (room) 5 and 9")
print(f'θss_Q = {np.around(θss_Q, 2)} °C')

With an outdoor temperature of 10 °C and a setpoint of 21 °C, both room air temperatures stay slightly below the setpoint (about 20.9 °C). A P-controller only delivers heat when there is a temperature difference, so a small steady-state offset always remains. The internal gains of 25 W raise the temperatures by only about 0.02 K, because the controller with $K_p = 1000$ W/K compensates for almost all of the additional heat by reducing its heating power. Most of the temperature drop across the exterior wall occurs in the insulation (node 0 to node 2).

## Steady-State Analysis from the State-Space Representation

The state-space model is

\begin{equation}
\dot\theta_C = A_s\,\theta_C + B_s\,u, \qquad y = C_s\,\theta_C + D_s\,u,
\end{equation}

where $\theta_C$ contains only the temperatures of the nodes with a capacity, and $u$ contains all inputs. In steady state ($\dot\theta_C = 0$), the output is

\begin{equation}
y = (-C_s A_s^{-1} B_s + D_s)\,u.
\end{equation}

The order of the inputs in $u$ is given by `us`: first the temperature sources of the branches (q0, q10, q11, q12, q13), then the heat flow sources of the nodes (θ0, θ4, θ5, θ6, θ8, θ9).

In [ ]:
## Create steady-state input vector
# temperature sources (order of us): q0, q10, q11, q12, q13

bT = np.array([To_ss, Ti_sp_ss, Ti_sp_ss, To_ss, To_ss])
# heat flow sources (order of us): θ0, θ4, θ5, θ6, θ8, θ9 (no solar radiation)
fQ = np.array([0, 0, Qa_ss, 0, 0, Qa_ss])
uss = np.hstack([bT, fQ])
uss = pd.Series(uss, index=us.index)
uss

In [ ]:
inv_As = pd.DataFrame(np.linalg.inv(As),
                      columns=As.index, index=As.index)
yss = (-Cs @ inv_As @ Bs + Ds) @ uss

print(f"Node/Room {yss.index[0]}: {yss.iloc[0]:.2f}")
print(f"Node/Room {yss.index[1]}: {yss.iloc[1]:.2f}")

In [ ]:
print(f"Error between DAE and state-space\n{abs(yss - θss_Q[[5, 9]])}")

The difference between the two methods is of the order of $10^{-14}$ °C, i.e. machine precision. This confirms that the state-space model is a correct transformation of the thermal network.

## Eigenvalue Analysis
### Time step

The eigenvalues $\lambda_i$ of the state matrix $A_s$ are negative, and their inverses define the time constants of the system, $T_i = -1/\lambda_i$. The explicit Euler method is only numerically stable if the time step is smaller than

\begin{equation}
\Delta t_{max} = 2 \min_i T_i.
\end{equation}

The time step `dt` is either imposed in the settings (`imposed_time_step = True`, `Δt`) or taken as $\Delta t_{max}$ rounded down. The implicit Euler method is unconditionally stable, so a time step larger than $\Delta t_{max}$ only makes the explicit solution unstable.

Without air capacities (`air_with_capacity = False`), the fast dynamics of the room air are removed from the model. This is why $\Delta t_{max}$ is several hours here; with air capacities it becomes much smaller.

In [ ]:
λ = np.linalg.eig(As)[0]  # eigenvalues of the state matrix

# time step
Δtmax = 2 * min(-1. / λ)  # max time step for stability of Euler explicit
db.print_rounded_time('Δtmax', Δtmax)

if imposed_time_step:
    dt = Δt
else:
    dt = db.round_time(Δtmax)

db.print_rounded_time('dt', dt)
print(f"dt = {dt:.0f} s")

### Settling time

The settling time is estimated as four times the largest time constant, $t_{settle} = 4 \max_i T_i$. After this time, the response to a step input has reached about 98 % of its final value. The slowest time constant is set by the concrete walls, so the system needs several days to settle.

`duration` is the settling time rounded up to the next full hour; it would be the length of a step-response simulation. The simulation below uses the period `start_date` – `end_date` from the settings instead.

In [ ]:
# settling time
t_settle = 4 * max(-1 / λ)
db.print_rounded_time('t_settle', t_settle)

# duration: next multiple of 3600 s that is larger than t_settle
# (not used for the simulation with weather data below)
duration = np.ceil(t_settle / 3600) * 3600
db.print_rounded_time('duration', duration)

## Simulation with Weather Data
### Input data

The model is driven by an hourly weather file in EPW format for Hombrechtikon (ZH). It is a typical meteorological year (TMYx, 2011–2025), i.e. a representative year assembled from months of different years. The air temperature, the direct normal radiation and the diffuse horizontal radiation are used. Because the months come from different years, all timestamps are mapped to the year 2000, and the data is cut to the simulation period defined in the settings.

In [ ]:
# read hourly weather data (EPW format)
filename = Path("weather_data/Hombrechtikon/CHE_ZH_Hombrechtikom.068510_TMYx.2011-2025.epw")
[data, meta] = db.read_epw(filename, coerce_year=None)
weather = data[["temp_air", "dir_n_rad", "dif_h_rad"]]

In [ ]:
# map all timestamps to the same year (the TMY combines months of different years)
weather.index = weather.index.map(lambda t: t.replace(year=2000))
# keep only the simulation period
weather = weather.loc[start_date:end_date]

### Solar radiation

The total solar irradiance $E_{tot}$ (direct, diffuse and ground-reflected) on a vertical surface is computed with `db.sol_rad_tilt_surf` from the weather data, the orientation of the surface (slope and azimuth) and the latitude of the site (47.24° N for Hombrechtikon, as given in the header of the weather file). Two orientations are needed: east for the exterior wall of Room 5 and north for the windows of both rooms.

The solar heat flow absorbed by the outer surface of the exterior wall is

\begin{equation}
\Phi_o = \alpha_{w,SW} \, S_w \, E_{tot,east},
\end{equation}

where $\alpha_{w,SW}$ is the short-wave absorptivity of the wall surface (`alpha`), $S_w$ is the wall area (`wall_area`) and $E_{tot,east}$ is the total solar irradiance on the east façade.

The solar radiation transmitted through a window is distributed to the interior surfaces according to the view factor between the window and each surface. Figure 3 shows a schematic representation of this distribution.

![thermal-network](images/Room_Plan2.png)

Figure 3: Schematic representation of the solar radiation entering through the windows and reaching the interior surfaces.

The short-wave solar radiation absorbed by each receiving interior surface (nodes 4 and 6 in Room 5, node 8 in Room 9) is

\begin{equation}
\Phi_i = F_{w,j} \, \tau \, S_w \, E_{tot,north},
\end{equation}

where $F_{w,j}$ is the [view factor](https://sterad.net/) between the window and the receiving surface $j$ (`view_factor`, the same for all surfaces), $\tau$ is the solar transmittance of the window (`beta_window`), $S_w$ is the window area and $E_{tot,north}$ is the total solar irradiance on the north façade. The part of the transmitted radiation that reaches the floor, the ceiling and the neglected walls is not included in the model.

In [ ]:
# vertical surfaces (slope = 90°); azimuth: 0° = south, -90° = east, 180° = north
surface_east = {'slope': slope,
                'azimuth': azimuth_east,
                'latitude': latitude}

surface_north = {'slope': slope,
                'azimuth': azimuth_north,
                'latitude': latitude}

# total irradiance (direct + diffuse + reflected) in W/m²
Etot_east = db.sol_rad_tilt_surf(
    weather, surface_east, albedo).sum(axis=1)

Etot_north = db.sol_rad_tilt_surf(
    weather, surface_north, albedo).sum(axis=1)

In [ ]:
# Create input_data_set
n = weather.shape[0]

To = weather["temp_air"]  # outdoor temperature
Ti_sp = temp_indor_setpoint * np.ones(n)  # indoor temperature set point
Φo = wall_area * alpha * Etot_east  # absorbed by the east wall (node 0)
Φi = wall_area * view_factor * beta_window * Etot_north  # absorbed by each interior surface (nodes 4, 6, 8)
Qa = Qa * np.ones(n)  # internal heat gains

# resample the hourly data to the time step dt (linear interpolation)
data = {'T_0': To, 'Ti_sp': Ti_sp, 'Φo': Φo, 'Φi': Φi, 'Qa': Qa}
input_data_set = pd.DataFrame(data, index=weather.index).resample(str(dt) + 'S').interpolate(method="linear")

# input vector u in time, ordered as us
u = db.inputs_in_time(us, input_data_set)

### Time integration

Starting from a uniform initial temperature (`temp_init`), the state equation is integrated in time with both the explicit and the implicit Euler method:

\begin{equation}
\text{explicit:} \quad \theta_{C,k+1} = (I + \Delta t\, A_s)\,\theta_{C,k} + \Delta t\, B_s\, u_k
\end{equation}

\begin{equation}
\text{implicit:} \quad \theta_{C,k+1} = (I - \Delta t\, A_s)^{-1}\,(\theta_{C,k} + \Delta t\, B_s\, u_k)
\end{equation}

The outputs are then obtained from $y = C_s\,\theta_C + D_s\,u$.

In [ ]:
# Initial conditions
θ_exp = pd.DataFrame(index=u.index)  # empty df with index for explicit Euler
θ_imp = pd.DataFrame(index=u.index)  # empty df with index for implicit Euler

θ_exp[As.columns] = temp_init  # fill θ for Euler explicit with initial values θ0
θ_imp[As.columns] = temp_init  # fill θ for Euler implicit with initial values θ0

I = np.eye(As.shape[0])

for k in tqdm(range(u.shape[0] - 1)):
    θ_exp.iloc[k + 1] = (I + dt * As) @ θ_exp.iloc[k] + dt * Bs @ u.iloc[k]
    θ_imp.iloc[k + 1] = np.linalg.inv(I - dt * As) @ (θ_imp.iloc[k] + dt * Bs @ u.iloc[k])

# outputs
y_exp = (Cs @ θ_exp.T + Ds @ u.T).T
y_imp = (Cs @ θ_imp.T + Ds @ u.T).T

### Results

In [ ]:
y = pd.concat([y_exp.iloc[:, 0], y_imp.iloc[:, 0]], axis=1, keys=['Explicit', 'Implicit'])
y.columns = y.columns.get_level_values(0)

# thin to at most ~5000 points
n = max(1, len(y) // 5000)
y_plot = y.iloc[::n]

fig, ax = plt.subplots()
t = y_plot.index.to_numpy()
ax.plot(t, y_plot['Explicit'].to_numpy(dtype=float), label='Explicit')
ax.plot(t, y_plot['Implicit'].to_numpy(dtype=float), label='Implicit')
ax.legend()
ax.set_xlabel('Time')
ax.set_ylabel('Indoor temperature, $\\theta_i$ / °C')
ax.set_title(f'Node/Room 5\nTime step: $dt$ = {dt:.0f} s; $dt_{{max}}$ = {Δtmax:.0f} s')
ax.set_ylim(19, 26)
fig.autofmt_xdate()
plt.show()

Figure 4: Indoor air temperature of Room 5 over the simulation period, computed with the explicit and the implicit Euler method.

Both methods give practically identical results, because the time step is smaller than $\Delta t_{max}$. The temperature stays within a few tenths of a degree of the setpoint of 21 °C. Because the controller is purely proportional, the deviation is directly linked to the HVAC power: $\theta - T_{i,sp} = -q_{HVAC}/K_p$. In winter, heat losses must be compensated by heating, so the room is slightly below 21 °C. In summer, solar gains and higher outdoor temperatures must be removed by cooling, so the room is up to about 0.5 °C above the setpoint. With a time step of 4 h, each day is resolved with only six points; a smaller time step can be imposed in the settings to resolve the daily cycle in more detail.

In [ ]:
y = pd.concat([y_exp.iloc[:, 1], y_imp.iloc[:, 1]], axis=1, keys=['Explicit', 'Implicit'])
y.columns = y.columns.get_level_values(0)

# thin to at most ~5000 points
n = max(1, len(y) // 5000)
y_plot = y.iloc[::n]

fig, ax = plt.subplots()
t = y_plot.index.to_numpy()
ax.plot(t, y_plot['Explicit'].to_numpy(dtype=float), label='Explicit')
ax.plot(t, y_plot['Implicit'].to_numpy(dtype=float), label='Implicit')
ax.legend()
ax.set_xlabel('Time')
ax.set_ylabel('Indoor temperature, $\\theta_i$ / °C')
ax.set_title(f'Node/Room 9\nTime step: $dt$ = {dt:.0f} s; $dt_{{max}}$ = {Δtmax:.0f} s')
ax.set_ylim(19, 26)
fig.autofmt_xdate()
plt.show()

Figure 5: Indoor air temperature of Room 9 over the simulation period, computed with the explicit and the implicit Euler method.

Room 9 shows the same seasonal behaviour as Room 5, but with smaller deviations from the setpoint in summer. Room 9 has no exterior wall, and the solar radiation entering through its window is absorbed by only one modelled surface (node 8). Room 5 receives transmitted radiation on two surfaces (nodes 4 and 6) and additionally has the sunlit east wall.

In [ ]:
# HVAC power of room 5 per m² floor area (l²): q = Kp (Ti_sp - θ5) / l²
# positive = heating, negative = cooling
Kp5 = TC["G"]["q11"]
q_HVAC_5 = Kp5 * (u["q11"] - y_imp["θ5"]) / l ** 2

# thin to at most ~5000 points
n = max(1, len(q_HVAC_5) // 5000)
q_plot = q_HVAC_5.iloc[::n]

fig, ax = plt.subplots()
ax.plot(q_plot.index.to_numpy(), q_plot.to_numpy(dtype=float))
ax.set_ylabel(r"Heating (+) / cooling (−) power $q$ / $(W \cdot m^{-2})$")
ax.set_xlabel(r"Time")
ax.set_title("Heating and cooling power in Room 5")
fig.autofmt_xdate()
plt.show()

Figure 6: Heating (positive) and cooling (negative) power of the HVAC system in Room 5 per square metre of floor area, computed from the implicit Euler solution as $q = K_p\,(T_{i,sp} - \theta_5)/l^2$.

Heating is needed only in winter and remains small (at most about 15 W/m²), thanks to the well-insulated exterior wall, the triple-glazed window and the internal heat gains. From spring to autumn cooling dominates, with peaks of about 50 W/m² in summer. The large cooling demand is mainly caused by the solar radiation transmitted through the large window (9 m², transmittance 0.8) and by outdoor temperatures above the setpoint. The strong short-term fluctuations reflect the day-night cycle and changing weather conditions. If the controller is switched off (`controller = False`), $K_p = 0$ and the power is zero.

## Conclusion

- A thermal network with 10 nodes and 14 branches was set up for two adjacent rooms and converted into a state-space model.
- The steady-state results of the DAE and of the state-space model agree to machine precision, which confirms that the model is implemented consistently.
- Without air capacities, the eigenvalue analysis gives a maximum time step of about 4.3 h for the explicit Euler method. With the chosen time step of 4 h, the explicit and implicit Euler methods give practically identical results.
- With a P-controller ($K_p = 1000$ W/K), both rooms are kept within about 0.5 °C of the setpoint over the whole year. The remaining deviation is the typical offset of a proportional controller.
- Room 5 needs little heating in winter but considerable cooling in summer, mainly because of the solar gains through the window. Room 9 shows the same behaviour with smaller deviations.
- The main limitations of the model are the simplified geometry (all surfaces 9 m², fully glazed façades), the adiabatic floor, ceiling and neighbouring walls, the missing ventilation and long-wave radiation, the constant internal gains and the large time step, which resolves the daily cycle only coarsely.

## Outlook: Extension of the Thermal Network to Include Long-Wave Radiation

In the thermal network considered in this work, long-wave radiative heat exchange is neglected. This effect can be included by accounting for the radiative exchange between the different walls and windows, which requires additional conductances in the thermal network. The updated network is shown in Figure 7. Additional conductances were added between nodes 0 and 5, 4 and 5, 6 and 5, and 8 and 9. These new conductances are highlighted in red.

![thermal-network](images/ThermalNetwork2.png)

Figure 7: Thermal network including long-wave radiation.

Each of these conductances consists of three conductances connected in series. The first and third conductances, $G_i$, describe the radiative resistance of the two surfaces and are calculated as

\begin{equation}
G_i = \frac{\epsilon_i}{1-\epsilon_i} S_i,
\end{equation}

where $\epsilon_i$ is the [emissivity](https://www.engineeringtoolbox.com/emissivity-coefficients-d_447.html#gsc.tab=0) and $S_i$ the area of surface $i$.

The second conductance, $G_k$, describes the geometric (view-factor) resistance between the two surfaces and is given by

\begin{equation}
G_k = F_k \cdot S_k,
\end{equation}

where $F_k$ is the [view factor](https://sterad.net/) between the surfaces and $S_k$ the corresponding surface area.

The three conductances are combined into a single equivalent conductance using the relation for conductances in series:

\begin{equation}
\frac{1}{G} = \frac{1}{G_{i,1}} + \frac{1}{G_{k}} + \frac{1}{G_{i,2}}.
\end{equation}

Radiative heat exchange depends on the fourth power of the absolute temperatures and is therefore non-linear. To express it in the same linear form as conduction and convection, it is linearized around the mean surface temperature by multiplying each conductance by the factor

\begin{equation}
k = 4 \cdot \sigma \cdot \bar{T}^3,
\end{equation}

where $\sigma$ is the Stefan-Boltzmann constant and $\bar{T}$ is the mean absolute temperature of the interacting surfaces. The resulting conductances (in W/K) can then be added directly to the thermal network.